# Pandas Labs – Limpieza, análisis y estructuración de datos
**GA1-220501094-01-AA1-EV06** · Proyecto: *Análisis de la siniestralidad vial en Colombia*

Este cuaderno continúa el diagnóstico de calidad de la EV05 (`data_quality.ipynb`) y desarrolla en orden:

| Lección | Tema | Laboratorio guiado | Reto |
|---|---|---|---|
| 2 | Limpieza y estandarización | Pipeline de limpieza con informe de cambios | Reto 2 (7 ejercicios) |
| 3 | Filtrado, ordenamiento y agrupación | Perfil de la siniestralidad vial (temporal y territorial) | Reto 3 (7 ejercicios) |
| 4 | Combinación de datasets | Tabla de análisis por municipio | Reto 4 (7 ejercicios) |

**Pregunta orientadora (adaptada al proyecto):** ¿Cómo preparar y estructurar los datos de siniestralidad vial para obtener información confiable sobre dónde y cuándo se concentran los lesionados y fallecidos?

> Equivalencias con el proyecto guía Instacart: *órdenes → registros de lesionados*, *productos → municipios*, *pasillos/departamentos → departamentos y sectores viales*, *hábitos de compra → patrones de siniestralidad*.

In [1]:
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

BASE = Path("..") if Path("../data").exists() else Path(".")
RAW = BASE / "data" / "raw"
PROCESSED = BASE / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

print("Pandas", pd.__version__)
sorted(p.name for p in RAW.glob("*.csv"))

Pandas 3.0.5


['lesiones_accidentes_transito.csv',
 'sectores_criticos_siniestralidad_vial.csv']

---
# Lección 2. Limpieza y estandarización
## 2.1 Ejemplos de la lección
Se cargan los datos **originales** (como en la EV05) para aplicar sobre ellos cada técnica.

In [2]:
lesiones_raw = pd.read_csv(RAW / "lesiones_accidentes_transito.csv")
sectores_raw = pd.read_csv(RAW / "sectores_criticos_siniestralidad_vial.csv")
print(lesiones_raw.shape, sectores_raw.shape)
lesiones_raw.dtypes

(644042, 6) (316, 13)


FECHA HECHO       str
COD_DEPTO       int64
DEPARTAMENTO      str
COD_MUNI        int64
MUNICIPIO         str
CANTIDAD        int64
dtype: object

### Ejemplo 1 – Renombrar columnas a `snake_case`

In [3]:
def a_snake_case(columnas):
    return (pd.Index(columnas).str.strip().str.lower()
            .str.replace(" ", "_", regex=False))

ej = sectores_raw.copy()
ej.columns = a_snake_case(ej.columns)
ej = ej.rename(columns={"gizscore": "gi_zscore", "gipvalue": "gi_pvalue"})
list(ej.columns)

['id_mt',
 'entidad',
 'gi_zscore',
 'fallecidos',
 'gi_pvalue',
 'tramo',
 'nombre',
 'latitud',
 'longitud',
 'pr',
 'municipio',
 'departamento',
 'divipola']

### Ejemplo 2 – Estandarizar texto: espacios, mayúsculas y tildes
Se quitan tildes pero se conserva la **Ñ** (NARIÑO no debe convertirse en NARINO).

In [4]:
TABLA_TILDES = str.maketrans("ÁÉÍÓÚÜáéíóúü", "AEIOUUaeiouu")

def normalizar_texto(serie):
    return (serie.astype("string")
                 .str.strip()
                 .str.upper()
                 .str.replace(r"\s+", " ", regex=True)
                 .str.translate(TABLA_TILDES))

pd.DataFrame({"original": sectores_raw["Departamento"].unique()[:8],
              "normalizado": normalizar_texto(pd.Series(sectores_raw["Departamento"].unique()[:8]))})

,original,normalizado
0,CUNDINAMARCA,CUNDINAMARCA
1,META,META
2,BOYACÁ,BOYACA
3,SANTANDER,SANTANDER
4,QUINDÍO,QUINDIO
5,TOLIMA,TOLIMA
6,CESAR,CESAR
7,RISARALDA,RISARALDA


### Ejemplo 3 – Convertir tipos: fechas, números con coma decimal y códigos con ceros

In [5]:
fecha = pd.to_datetime(lesiones_raw["FECHA HECHO"], format="%d/%m/%Y", errors="coerce")
zscore = pd.to_numeric(sectores_raw["GiZScore"].str.replace(",", ".", regex=False), errors="coerce")
cod_muni = lesiones_raw["COD_MUNI"].astype(str).str.zfill(5)

print("Fecha  :", lesiones_raw["FECHA HECHO"].dtype, "->", fecha.dtype, "| NaT:", fecha.isna().sum())
print("ZScore :", sectores_raw["GiZScore"].dtype, "->", zscore.dtype, "| NaN:", zscore.isna().sum())
print("Código :", lesiones_raw["COD_MUNI"].iloc[85], "->", cod_muni.iloc[85])

Fecha  : str -> datetime64[us] | NaT: 0
ZScore : str -> float64 | NaN: 0
Código : 5045 -> 05045


### Ejemplo 4 – Valores faltantes ocultos → `NaN`, y tratamiento con `fillna` / `dropna`

In [6]:
ej = sectores_raw[["ID_MT", "PR", "Nombre"]].replace(["<Null>", "sd"], np.nan)
print("Nulos después de reemplazar marcadores:"); print(ej.isna().sum())
print("\nfillna('SIN NOMBRE') ->", ej["Nombre"].fillna("SIN NOMBRE").isna().sum(), "nulos")
print("dropna(subset=['PR']) ->", len(ej.dropna(subset=["PR"])), "de", len(ej), "filas")

Nulos después de reemplazar marcadores:
ID_MT     175
PR         42
Nombre    175
dtype: int64

fillna('SIN NOMBRE') -> 0 nulos
dropna(subset=['PR']) -> 274 de 316 filas


### Ejemplo 5 – Duplicados: completos y por subconjunto de columnas

In [7]:
print("Filas idénticas (lesiones):", lesiones_raw.duplicated().sum())
print("Coordenadas repetidas (sectores):", sectores_raw.duplicated(subset=["Latitud", "Longitud"]).sum())
print("drop_duplicates por coordenadas:", len(sectores_raw), "->",
      len(sectores_raw.drop_duplicates(subset=["Latitud", "Longitud"])))

Filas idénticas (lesiones): 294354
Coordenadas repetidas (sectores): 4
drop_duplicates por coordenadas: 316 -> 312


### Ejemplo 6 – Valores atípicos con rango intercuartílico (IQR)

In [8]:
f = pd.to_numeric(sectores_raw["Fallecidos"])
q1, q3 = f.quantile([0.25, 0.75])
iqr = q3 - q1
lim_sup = q3 + 1.5 * iqr
print(f"Q1={q1}, Q3={q3}, IQR={iqr}, límite superior={lim_sup}")
print("Sectores con fallecidos atípicos:", (f > lim_sup).sum())

Q1=7.0, Q3=14.0, IQR=7.0, límite superior=24.5
Sectores con fallecidos atípicos: 15


## 2.2 Laboratorio guiado 2 – Pipeline de limpieza con informe de cambios
Cada paso es una función que recibe un DataFrame, devuelve uno nuevo y **registra** lo que cambió (filas antes/después, valores modificados y descripción). Al final se obtiene un informe de cambios auditable.

In [9]:
informe_cambios = []

def registrar(dataset, paso, df_antes, df_despues, valores_afectados, descripcion):
    informe_cambios.append({
        "dataset": dataset, "paso": paso,
        "filas_antes": len(df_antes), "filas_despues": len(df_despues),
        "valores_afectados": int(valores_afectados), "descripcion": descripcion,
    })
    return df_despues

MARCADORES_NULOS = ["SIN ESTABLECER", "<Null>", "sd"]
NOMBRE_MES = {1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril", 5: "Mayo", 6: "Junio", 7: "Julio",
              8: "Agosto", 9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"}
NOMBRE_DIA = {0: "Lunes", 1: "Martes", 2: "Miércoles", 3: "Jueves", 4: "Viernes", 5: "Sábado", 6: "Domingo"}

### Pipeline para `lesiones`

In [10]:
def limpiar_lesiones(raw):
    d = "lesiones"
    # 1. Renombrar columnas
    df = raw.rename(columns={"FECHA HECHO": "fecha", "COD_DEPTO": "cod_depto", "DEPARTAMENTO": "departamento",
                             "COD_MUNI": "cod_muni", "MUNICIPIO": "municipio", "CANTIDAD": "cantidad"})
    df = registrar(d, "1. Renombrar columnas", raw, df, raw.shape[1], "Nombres a snake_case en español")

    # 2. Fecha a datetime
    antes = df; df = df.copy()
    df["fecha"] = pd.to_datetime(df["fecha"], format="%d/%m/%Y", errors="coerce")
    df = registrar(d, "2. Convertir fecha", antes, df, len(df), f"texto -> datetime64; NaT: {df['fecha'].isna().sum()}")

    # 3. Códigos DIVIPOLA como texto con ceros
    antes = df; df = df.copy()
    afectados = (df["cod_muni"] < 10000).sum()
    df["cod_depto"] = df["cod_depto"].astype(str).str.zfill(2)
    df["cod_muni"] = df["cod_muni"].astype(str).str.zfill(5)
    df = registrar(d, "3. Códigos DIVIPOLA", antes, df, afectados, "int -> texto con zfill(2)/zfill(5)")

    # 4. Normalizar texto
    antes = df; df = df.copy()
    for c in ["departamento", "municipio"]:
        df[c] = normalizar_texto(df[c])
    cambios = (antes[["departamento", "municipio"]].astype("string") != df[["departamento", "municipio"]]).sum().sum()
    df = registrar(d, "4. Normalizar texto", antes, df, cambios, "strip, mayúsculas, espacios y tildes")

    # 5. Nulos ocultos 'SIN ESTABLECER' -> NaN (también su código)
    antes = df; df = df.copy()
    sin_depto = df["departamento"].isin(MARCADORES_NULOS)
    sin_muni = df["municipio"].isin(MARCADORES_NULOS)
    df.loc[sin_depto, ["departamento", "cod_depto"]] = pd.NA
    df.loc[sin_muni, ["municipio", "cod_muni"]] = pd.NA
    df = registrar(d, "5. Nulos ocultos", antes, df, sin_depto.sum() + sin_muni.sum(), "'SIN ESTABLECER' -> NaN")

    # 6. Variables derivadas de la fecha
    antes = df; df = df.copy()
    df["anio"] = df["fecha"].dt.year
    df["mes"] = df["fecha"].dt.month
    df["nombre_mes"] = df["mes"].map(NOMBRE_MES)
    df["dia_semana"] = df["fecha"].dt.dayofweek.map(NOMBRE_DIA)
    df["anio_parcial"] = df["anio"] == df["fecha"].max().year   # 2026 llega hasta septiembre
    df = registrar(d, "6. Variables de fecha", antes, df, 5, "anio, mes, nombre_mes, dia_semana, anio_parcial")

    # 7. Marcar atípicos (no se eliminan: son eventos reales con muchos lesionados)
    antes = df; df = df.copy()
    p999 = df["cantidad"].quantile(0.999)
    df["cantidad_atipica"] = df["cantidad"] > p999
    df = registrar(d, "7. Marcar atípicos", antes, df, df["cantidad_atipica"].sum(), f"cantidad > percentil 99,9 ({p999:.0f})")

    # 8. Duplicados: se conservan (no hay ID de víctima), solo se documentan
    df = registrar(d, "8. Duplicados", df, df, df.duplicated().sum(), "Se conservan: sin ID de víctima no son errores")

    # 9. Tipos categóricos para ahorrar memoria
    antes = df; df = df.copy()
    for c in ["departamento", "municipio", "dia_semana", "nombre_mes"]:
        df[c] = df[c].astype("category")
    df = registrar(d, "9. Tipos categóricos", antes, df, 4, "texto repetitivo -> category")
    return df

lesiones = limpiar_lesiones(lesiones_raw)
lesiones.head()

,fecha,cod_depto,departamento,cod_muni,municipio,cantidad,anio,mes,nombre_mes,dia_semana,anio_parcial,cantidad_atipica
0,2026-09-30,76,VALLE DEL CAUCA,76001,CALI,1,2026,9,Septiembre,Miércoles,True,False
1,2026-09-30,76,VALLE DEL CAUCA,76001,CALI,1,2026,9,Septiembre,Miércoles,True,False
2,2026-09-30,76,VALLE DEL CAUCA,76001,CALI,1,2026,9,Septiembre,Miércoles,True,False
3,2026-09-30,76,VALLE DEL CAUCA,76001,CALI,1,2026,9,Septiembre,Miércoles,True,False
4,2026-09-30,70,SUCRE,70001,SINCELEJO,1,2026,9,Septiembre,Miércoles,True,False


### Pipeline para `sectores`

In [11]:
def limpiar_sectores(raw):
    d = "sectores"
    # 1. Renombrar columnas
    df = raw.copy()
    df.columns = a_snake_case(df.columns)
    df = df.rename(columns={"gizscore": "gi_zscore", "gipvalue": "gi_pvalue", "divipola": "cod_muni"})
    df = registrar(d, "1. Renombrar columnas", raw, df, raw.shape[1], "snake_case; divipola -> cod_muni")

    # 2. Nulos ocultos
    antes = df
    afectados = df.isin(MARCADORES_NULOS).sum().sum()
    df = df.replace(MARCADORES_NULOS, np.nan)
    df = registrar(d, "2. Nulos ocultos", antes, df, afectados, "'<Null>' y 'sd' -> NaN")

    # 3. Coma decimal -> float y PR numérico
    antes = df; df = df.copy()
    for c in ["gi_zscore", "gi_pvalue", "latitud", "longitud"]:
        df[c] = pd.to_numeric(df[c].str.replace(",", ".", regex=False))
    df["pr"] = pd.to_numeric(df["pr"])
    df = registrar(d, "3. Tipos numéricos", antes, df, 5 * len(df), "gi_zscore, gi_pvalue, latitud, longitud, pr -> float")

    # 4. Código DIVIPOLA
    antes = df; df = df.copy()
    afectados = (df["cod_muni"] < 10000).sum()
    df["cod_muni"] = df["cod_muni"].astype(str).str.zfill(5)
    df["cod_depto"] = df["cod_muni"].str[:2]
    df = registrar(d, "4. Código DIVIPOLA", antes, df, afectados, "cod_muni zfill(5) y nuevo cod_depto")

    # 5. Normalizar texto
    antes = df; df = df.copy()
    for c in ["departamento", "municipio", "tramo", "nombre", "entidad"]:
        df[c] = normalizar_texto(df[c])
    df["tramo"] = df["tramo"].str.replace("*", "", regex=False).str.strip()
    cambios = sum((antes[c].astype("string") != df[c]).fillna(False).sum() for c in ["departamento", "municipio", "tramo", "nombre"])
    df = registrar(d, "5. Normalizar texto", antes, df, cambios, "tildes, mayúsculas, espacios dobles, '*' en tramo")

    # 6. Imputar descriptivos faltantes
    antes = df; df = df.copy()
    afectados = df[["tramo", "nombre"]].isna().sum().sum()
    df["tramo"] = df["tramo"].fillna("SIN TRAMO")
    df["nombre"] = df["nombre"].fillna("SIN NOMBRE")
    df = registrar(d, "6. Imputar texto faltante", antes, df, afectados, "tramo/nombre nulos -> 'SIN TRAMO'/'SIN NOMBRE'")

    # 7. Duplicados por coordenadas: conservar el de mayor gi_zscore
    antes = df
    df = (df.sort_values("gi_zscore", ascending=False)
            .drop_duplicates(subset=["latitud", "longitud"], keep="first")
            .sort_index())
    df = registrar(d, "7. Duplicados por coordenadas", antes, df, len(antes) - len(df), "Se conserva el punto con mayor gi_zscore")

    # 8. Nivel de significancia del punto caliente
    antes = df; df = df.copy()
    df["nivel_confianza"] = pd.cut(df["gi_pvalue"], bins=[-0.001, 0.01, 0.05, 0.10],
                                   labels=["99%", "95%", "90%"])
    df = registrar(d, "8. Nivel de confianza", antes, df, len(df), "gi_pvalue -> 99% / 95% / 90%")

    # 9. Tipos categóricos
    antes = df; df = df.copy()
    df["entidad"] = df["entidad"].astype("category")
    df = registrar(d, "9. Tipos categóricos", antes, df, 1, "entidad -> category")
    return df.reset_index(drop=True)

sectores = limpiar_sectores(sectores_raw)
sectores.head()

,id_mt,entidad,gi_zscore,fallecidos,gi_pvalue,tramo,nombre,latitud,longitud,pr,municipio,departamento,cod_muni,cod_depto,nivel_confianza
0,4G013,ANI,1.77,11,0.08,BOGOTA - VILLAVICENCIO,IP – CHIRAJARA - FUNDADORES,4.45,-74.05,10.00,CHIPAQUE,CUNDINAMARCA,25178,25,90%
1,4G013,ANI,3.82,23,0.00,BOGOTA - VILLAVICENCIO,IP – CHIRAJARA - FUNDADORES,4.23,-73.82,58.00,GUAYABETAL,CUNDINAMARCA,25335,25,99%
2,4G013,ANI,2.80,1,0.01,BOGOTA - VILLAVICENCIO,IP – CHIRAJARA - FUNDADORES,4.20,-73.81,60.00,GUAYABETAL,CUNDINAMARCA,25335,25,99%
3,4G013,ANI,1.73,1,0.08,BOGOTA - VILLAVICENCIO,IP – CHIRAJARA - FUNDADORES,4.12,-73.64,85.00,VILLAVICENCIO,META,50001,50,90%
4,NaN,INVIAS-OTROS,1.73,13,0.08,GRANADA - VILLAVICENCIO,SIN NOMBRE,4.12,-73.65,NaN,VILLAVICENCIO,META,50001,50,90%


### Informe de cambios

In [12]:
reporte_limpieza = pd.DataFrame(informe_cambios)
reporte_limpieza

,dataset,paso,filas_antes,filas_despues,valores_afectados,descripcion
0,lesiones,1. Renombrar columnas,644042,644042,6,Nombres a snake_case en español
1,lesiones,2. Convertir fecha,644042,644042,644042,texto -> datetime64; NaT: 0
2,lesiones,3. Códigos DIVIPOLA,644042,644042,80832,int -> texto con zfill(2)/zfill(5)
3,lesiones,4. Normalizar texto,644042,644042,585,"strip, mayúsculas, espacios y tildes"
4,lesiones,5. Nulos ocultos,644042,644042,14,'SIN ESTABLECER' -> NaN
5,lesiones,6. Variables de fecha,644042,644042,5,"anio, mes, nombre_mes, dia_semana, anio_parcial"
6,lesiones,7. Marcar atípicos,644042,644042,638,"cantidad > percentil 99,9 (21)"
7,lesiones,8. Duplicados,644042,644042,294354,Se conservan: sin ID de víctima no son errores
8,lesiones,9. Tipos categóricos,644042,644042,4,texto repetitivo -> category
9,sectores,1. Renombrar columnas,316,316,13,snake_case; divipola -> cod_muni


In [13]:
print("Tipos finales – lesiones"); print(lesiones.dtypes, "\n")
print("Tipos finales – sectores"); print(sectores.dtypes)

Tipos finales – lesiones
fecha               datetime64[us]
cod_depto                      str
departamento              category
cod_muni                       str
municipio                 category
cantidad                     int64
anio                         int32
mes                          int32
nombre_mes                category
dia_semana                category
anio_parcial                  bool
cantidad_atipica              bool
dtype: object 

Tipos finales – sectores
id_mt                   str
entidad            category
gi_zscore           float64
fallecidos            int64
gi_pvalue           float64
tramo                string
nombre               string
latitud             float64
longitud            float64
pr                  float64
municipio            string
departamento         string
cod_muni                str
cod_depto               str
nivel_confianza    category
dtype: object


In [14]:
print("Nulos restantes – lesiones:"); print(lesiones.isna().sum()[lambda s: s > 0], "\n")
print("Nulos restantes – sectores:"); print(sectores.isna().sum()[lambda s: s > 0])

Nulos restantes – lesiones:
cod_depto        3
departamento     3
cod_muni        11
municipio       11
dtype: int64 

Nulos restantes – sectores:
id_mt    175
pr        42
dtype: int64


**Lectura del informe:** en `lesiones` no se elimina ninguna fila (644.042 antes y después). Los cambios son de **tipo y formato**, más 14 ubicaciones `SIN ESTABLECER` que pasan a nulo. En `sectores` se eliminan 4 puntos con coordenadas repetidas (316 → 312). Los únicos nulos que quedan son esperables: `id_mt` (vías sin contrato de concesión) y `pr` (sin punto de referencia).

## 2.3 Reto 2 – 7 ejercicios

**Ejercicio 2.1.** Compare los valores faltantes de `sectores` antes y después de la limpieza (incluidos los nulos ocultos).

In [15]:
antes_ocultos = sectores_raw.isna().sum() + sectores_raw.isin(MARCADORES_NULOS).sum()
antes_ocultos.index = a_snake_case(antes_ocultos.index)
antes_ocultos = antes_ocultos.rename({"gizscore": "gi_zscore", "gipvalue": "gi_pvalue", "divipola": "cod_muni"})
comparacion = pd.DataFrame({"antes": antes_ocultos, "despues": sectores.isna().sum()}).fillna(0).astype(int)
comparacion[(comparacion["antes"] > 0) | (comparacion["despues"] > 0)]

,antes,despues
id_mt,175,175
nombre,175,0
pr,42,42
tramo,23,0


**Ejercicio 2.2.** Verifique que, después de normalizar, todos los departamentos de `sectores` coinciden con los de `lesiones`.

In [16]:
dep_les = set(lesiones["departamento"].dropna().unique())
dep_sec_raw = set(sectores_raw["Departamento"].unique())
dep_sec = set(sectores["departamento"].unique())
print("Sin coincidencia ANTES  :", sorted(dep_sec_raw - set(lesiones_raw["DEPARTAMENTO"])))
print("Sin coincidencia DESPUÉS:", sorted(dep_sec - dep_les))

Sin coincidencia ANTES  : ['ATLÁNTICO', 'BOLÍVAR', 'BOYACÁ', 'CAQUETÁ', 'CÓRDOBA', 'QUINDÍO']
Sin coincidencia DESPUÉS: []


**Ejercicio 2.3.** Compruebe que las columnas de fecha derivadas son coherentes: sin `NaT`, meses de 1 a 12 y el número de días por año.

In [17]:
print("NaT en fecha:", lesiones["fecha"].isna().sum())
print("Meses:", sorted(lesiones["mes"].unique().tolist()))
print("Año parcial:", lesiones.loc[lesiones["anio_parcial"], "anio"].unique(),
      "| último mes:", lesiones.loc[lesiones["anio_parcial"], "mes"].max())
lesiones.groupby("anio")["fecha"].nunique().rename("dias_con_registro").to_frame().T

NaT en fecha: 0
Meses: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Año parcial: [2026] | último mes: 9


anio,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
dias_con_registro,365,366,365,365,365,366,365,365,365,366,365,365,365,366,365,365,365,366,365,365,365,366,365,273


**Ejercicio 2.4.** Compare dos criterios de atípicos para `cantidad`: IQR y percentil 99,9. Explique cuál conviene.

In [18]:
q1, q3 = lesiones["cantidad"].quantile([0.25, 0.75])
lim_iqr = q3 + 1.5 * (q3 - q1)
p999 = lesiones["cantidad"].quantile(0.999)
print(f"IQR: Q1={q1}, Q3={q3} -> límite {lim_iqr} -> atípicos: {(lesiones['cantidad'] > lim_iqr).sum():,}")
print(f"P99.9: límite {p999:.0f} -> atípicos: {(lesiones['cantidad'] > p999).sum():,}")
lesiones.loc[lesiones["cantidad_atipica"], ["fecha", "municipio", "cantidad"]].nlargest(5, "cantidad")

IQR: Q1=1.0, Q3=1.0 -> límite 1.0 -> atípicos: 79,726
P99.9: límite 21 -> atípicos: 638


,fecha,municipio,cantidad
422749,2016-04-25,BOGOTA D.C.,91
380444,2017-12-02,BOGOTA D.C.,63
421206,2016-05-16,BOGOTA D.C.,60
378776,2017-12-29,CALI,56
508731,2011-01-01,MEDELLIN,53


> Como el 75 % de los registros tiene `cantidad = 1`, el IQR es 0 y marcaría como atípico **cualquier** registro con más de un lesionado (≈12 %). El percentil 99,9 es más razonable: marca solo eventos excepcionales, que se conservan y se señalan.

**Ejercicio 2.5.** Cuantifique los duplicados y demuestre que eliminarlos alteraría el total de lesionados.

In [19]:
total_real = lesiones["cantidad"].sum()
sin_dup = lesiones.drop_duplicates()
print(f"Duplicados exactos: {lesiones.duplicated().sum():,}")
print(f"Total lesionados conservando filas: {total_real:,}")
print(f"Total si se eliminaran duplicados : {sin_dup['cantidad'].sum():,}  "
      f"(pérdida {1 - sin_dup['cantidad'].sum() / total_real:.1%})")

Duplicados exactos: 294,354
Total lesionados conservando filas: 885,829
Total si se eliminaran duplicados : 589,932  (pérdida 33.4%)


**Ejercicio 2.6.** Reemplace las siglas de `entidad` por nombres descriptivos con `map()`.

In [20]:
ENTIDADES = {"ANI": "Agencia Nacional de Infraestructura", "INVIAS-OTROS": "INVÍAS y otras vías nacionales",
             "DEPTO": "Vía departamental", "CIUDAD": "Vía urbana"}
sectores["entidad_nombre"] = sectores["entidad"].astype(str).map(ENTIDADES).astype("category")
print("Sin mapear:", sectores["entidad_nombre"].isna().sum())
sectores["entidad_nombre"].value_counts()

Sin mapear: 0


entidad_nombre
Agencia Nacional de Infraestructura    137
INVÍAS y otras vías nacionales         126
Vía departamental                       48
Vía urbana                               1
Name: count, dtype: int64

**Ejercicio 2.7.** Mida el ahorro de memoria obtenido con los tipos `category`.

In [21]:
mem_raw = lesiones_raw.memory_usage(deep=True).sum() / 1024**2
sin_cat = lesiones.astype({c: "string" for c in ["departamento", "municipio", "dia_semana", "nombre_mes"]})
mem_str = sin_cat.memory_usage(deep=True).sum() / 1024**2
mem_cat = lesiones.memory_usage(deep=True).sum() / 1024**2
pd.DataFrame({"MB": [mem_raw, mem_str, mem_cat]},
             index=["original (6 columnas)", "limpio con texto", "limpio con category"])

,MB
original (6 columnas),122.76
limpio con texto,224.73
limpio con category,83.59


### Guardar los datos limpios

In [22]:
lesiones.to_csv(PROCESSED / "lesiones_clean.csv", index=False)
sectores.to_csv(PROCESSED / "sectores_clean.csv", index=False)
reporte_limpieza.to_csv(PROCESSED / "cleaning_log.csv", index=False)
sorted(p.name for p in PROCESSED.glob("*.csv"))

['cleaning_log.csv',
 'dim_departments.csv',
 'dim_municipalities.csv',
 'lesiones_by_department_year.csv',
 'lesiones_by_year.csv',
 'lesiones_clean.csv',
 'municipality_analysis.csv',
 'sectores_clean.csv']

---
# Lección 3. Filtrado, ordenamiento y agrupación
## 3.1 Ejemplos de la lección
Se trabaja con los DataFrames limpios (`lesiones`, `sectores`). Para comparar años se excluye 2026 (incompleto) salvo cuando se compara el mismo periodo.

### Ejemplo 1 – Filtros booleanos con `&`, `|` y `~`

In [23]:
bogota_2025 = lesiones[(lesiones["municipio"] == "BOGOTA D.C.") & (lesiones["anio"] == 2025)]
print("Registros Bogotá 2025:", len(bogota_2025), "| lesionados:", bogota_2025["cantidad"].sum())
fin_semana = lesiones[lesiones["dia_semana"].isin(["Sábado", "Domingo"])]
print("Lesionados en fin de semana:", fin_semana["cantidad"].sum())

Registros Bogotá 2025: 3770 | lesionados: 3857
Lesionados en fin de semana: 295187


### Ejemplo 2 – `isin()`, `between()` y `query()`

In [24]:
caribe = ["ATLANTICO", "BOLIVAR", "CESAR", "CORDOBA", "LA GUAJIRA", "MAGDALENA", "SUCRE"]
print("Región Caribe:", lesiones[lesiones["departamento"].isin(caribe)]["cantidad"].sum())
print("2020-2022:", lesiones[lesiones["anio"].between(2020, 2022)]["cantidad"].sum())
sectores.query("fallecidos >= 20 and nivel_confianza == '99%'")[["tramo", "municipio", "fallecidos", "gi_zscore"]]

Región Caribe: 126660
2020-2022: 107355


,tramo,municipio,fallecidos,gi_zscore
1,BOGOTA - VILLAVICENCIO,GUAYABETAL,23,3.82
27,BOGOTA - MOSQUERA,FUNZA,25,6.55
57,BARRANCA - AGUACHICA,SAN ALBERTO,24,6.25
64,BARBOSA - BUCARAMANGA,FLORIDABLANCA,22,5.66
78,CALARCA - IBAGUE,CALARCA,21,4.51
85,MEDELLIN - SANTUARIO,BELLO,20,4.20
95,EL HATILLO - GIRARDOTA - COPACABANA - BELLO - MED,COPACABANA,29,5.53
117,BUGA- LA VICTORIA,SAN PEDRO,23,5.95
132,CALI - PALMIRA,PALMIRA,27,7.14
133,CALI - PALMIRA,PALMIRA,20,5.07


### Ejemplo 3 – Ordenar con `sort_values()` y `nlargest()`

In [25]:
sectores.sort_values(["departamento", "fallecidos"], ascending=[True, False])[
    ["departamento", "municipio", "tramo", "fallecidos"]].head(8)

,departamento,municipio,tramo,fallecidos
201,ANTIOQUIA,TURBO,CHIGORODO-TURBO,30
95,ANTIOQUIA,COPACABANA,EL HATILLO - GIRARDOTA - COPACABANA - BELLO - MED,29
202,ANTIOQUIA,APARTADO,CHIGORODO-TURBO,24
204,ANTIOQUIA,CAREPA,CHIGORODO-TURBO,24
85,ANTIOQUIA,BELLO,MEDELLIN - SANTUARIO,20
203,ANTIOQUIA,APARTADO,CHIGORODO-TURBO,20
87,ANTIOQUIA,BELLO,EL HATILLO - GIRARDOTA - COPACABANA - BELLO - MED,18
207,ANTIOQUIA,CHIGORODO,SANTAFE DE ANTIOQUIA - CHIGORODO,16


### Ejemplo 4 – `groupby()` con agregaciones con nombre

In [26]:
lesiones.groupby("departamento", observed=True).agg(
    registros=("cantidad", "size"),
    lesionados=("cantidad", "sum"),
    max_evento=("cantidad", "max"),
    municipios=("cod_muni", "nunique"),
).sort_values("lesionados", ascending=False).head(10)

,registros,lesionados,max_evento,municipios
departamento,,,,
VALLE DEL CAUCA,84498,121456,56,42
BOGOTA D.C.,64044,93841,91,1
ANTIOQUIA,58109,82634,53,124
SANTANDER,48429,72315,28,87
CUNDINAMARCA,46709,56147,27,116
RISARALDA,33085,51571,52,14
TOLIMA,33417,47096,36,47
ATLANTICO,22720,34200,29,23
CALDAS,27143,34135,14,27


### Ejemplo 5 – `value_counts()`, `pivot_table()` y `crosstab()`

In [27]:
print(sectores["nivel_confianza"].value_counts(normalize=True).round(3), "\n")
pd.crosstab(sectores["entidad"], sectores["nivel_confianza"], margins=True)

nivel_confianza
99%   0.53
95%   0.28
90%   0.19
Name: proportion, dtype: float64 



nivel_confianza,99%,95%,90%,All
entidad,,,,
ANI,64,37,36,137
CIUDAD,1,0,0,1
DEPTO,31,13,4,48
INVIAS-OTROS,70,36,20,126
All,166,86,60,312


### Ejemplo 6 – `transform()` para comparar cada grupo contra su total

In [28]:
ej = lesiones.groupby(["departamento", "anio"], observed=True)["cantidad"].sum().reset_index()
ej["pct_del_departamento"] = ej["cantidad"] / ej.groupby("departamento", observed=True)["cantidad"].transform("sum") * 100
ej[ej["departamento"] == "ANTIOQUIA"].tail(5)

,departamento,anio,cantidad,pct_del_departamento
43,ANTIOQUIA,2022,4254,5.15
44,ANTIOQUIA,2023,5834,7.06
45,ANTIOQUIA,2024,4363,5.28
46,ANTIOQUIA,2025,4697,5.68
47,ANTIOQUIA,2026,3491,4.22


## 3.2 Laboratorio guiado 3 – Perfil de la siniestralidad vial
*(Equivalente al “perfil de hábitos de compra” de Instacart: cuándo, dónde y con qué intensidad ocurren los lesionados.)*

Se responden cinco preguntas: evolución anual, estacionalidad mensual, día de la semana, concentración territorial y perfil de los sectores críticos.

In [29]:
completos = lesiones[~lesiones["anio_parcial"]]          # 2003-2025

# a) Evolución anual
perfil_anual = completos.groupby("anio").agg(lesionados=("cantidad", "sum"), registros=("cantidad", "size"))
perfil_anual["variacion_%"] = perfil_anual["lesionados"].pct_change() * 100
perfil_anual.tail(8)

,lesionados,registros,variacion_%
anio,,,
2018,48999,48999,-27.17
2019,40150,40150,-18.06
2020,27481,27481,-31.55
2021,34114,34114,24.14
2022,45760,45760,34.14
2023,51338,51338,12.19
2024,47344,45294,-7.78
2025,48432,46194,2.30


In [30]:
# b) Estacionalidad: promedio de lesionados por mes (2018-2025)
recientes = completos[completos["anio"] >= 2018]
perfil_mes = (recientes.groupby(["anio", "mes"])["cantidad"].sum()
              .groupby("mes").mean().rename("promedio_lesionados").to_frame())
perfil_mes["mes_nombre"] = perfil_mes.index.map(NOMBRE_MES)
perfil_mes.sort_values("promedio_lesionados", ascending=False).head(4)

,promedio_lesionados,mes_nombre
mes,,
3,"3,937.00",Marzo
10,"3,726.25",Octubre
9,"3,720.12",Septiembre
8,"3,704.75",Agosto


In [31]:
# c) Día de la semana
orden_dias = list(NOMBRE_DIA.values())
perfil_dia = (recientes.groupby("dia_semana", observed=True)["cantidad"].sum()
              .reindex(orden_dias).to_frame("lesionados"))
perfil_dia["%"] = perfil_dia["lesionados"] / perfil_dia["lesionados"].sum() * 100
perfil_dia

,lesionados,%
dia_semana,,
Lunes,48069,13.99
Martes,44590,12.98
Miércoles,44462,12.94
Jueves,44128,12.84
Viernes,47891,13.94
Sábado,56174,16.35
Domingo,58304,16.97


In [32]:
# d) Concentración territorial: departamentos 2018-2025 y % acumulado
perfil_depto = (recientes.groupby("departamento", observed=True)["cantidad"].sum()
                .sort_values(ascending=False).to_frame("lesionados"))
perfil_depto["%"] = perfil_depto["lesionados"] / perfil_depto["lesionados"].sum() * 100
perfil_depto["%_acumulado"] = perfil_depto["%"].cumsum()
perfil_depto.head(10)

,lesionados,%,%_acumulado
departamento,,,
VALLE DEL CAUCA,49903,14.52,14.52
ANTIOQUIA,37180,10.82,25.34
BOGOTA D.C.,32275,9.39,34.74
CUNDINAMARCA,25253,7.35,42.08
SANTANDER,22993,6.69,48.78
TOLIMA,22143,6.44,55.22
RISARALDA,18382,5.35,60.57
META,13285,3.87,64.44
CALDAS,12538,3.65,68.08


In [33]:
# e) Perfil de sectores críticos por departamento
perfil_sectores = (sectores.groupby("departamento")
                   .agg(sectores=("cod_muni", "size"), fallecidos=("fallecidos", "sum"),
                        zscore_max=("gi_zscore", "max"), municipios=("cod_muni", "nunique"))
                   .sort_values("fallecidos", ascending=False))
perfil_sectores.head(8)

,sectores,fallecidos,zscore_max,municipios
departamento,,,,
VALLE DEL CAUCA,60,684,8.91,16
CUNDINAMARCA,41,422,7.75,18
ANTIOQUIA,33,333,8.03,9
CESAR,22,293,6.25,12
NARIÑO,19,244,14.54,5
SANTANDER,17,201,5.66,8
CORDOBA,14,184,11.87,8
CAUCA,19,145,3.88,8


In [34]:
# Tabla dinámica departamento x año (top 5 departamentos, últimos 5 años completos)
top5 = perfil_depto.head(5).index
pd.pivot_table(completos[completos["departamento"].isin(top5) & (completos["anio"] >= 2021)],
               index="departamento", columns="anio", values="cantidad", aggfunc="sum", observed=True)

anio,2021,2022,2023,2024,2025
departamento,,,,,
ANTIOQUIA,2927,4254,5834,4363,4697
BOGOTA D.C.,3505,4378,5595,4734,3857
CUNDINAMARCA,2056,3525,4245,3943,4115
SANTANDER,2761,2787,3182,2879,2738
VALLE DEL CAUCA,5671,6655,6424,6160,5999


In [35]:
# Resumen numérico del perfil para la interpretación
print("Pico anual:", perfil_anual["lesionados"].idxmax(), f"({perfil_anual['lesionados'].max():,})")
print("Mes con más lesionados (promedio 2018-2025):", NOMBRE_MES[perfil_mes['promedio_lesionados'].idxmax()])
print("Día con más lesionados:", perfil_dia["lesionados"].idxmax(), f"({perfil_dia['%'].max():.1f} %)")
print("Departamentos que suman el 50 %:", (perfil_depto["%_acumulado"] <= 50).sum() + 1)
print("Departamento con más fallecidos en sectores críticos:", perfil_sectores.index[0])

Pico anual: 2016 (77,037)
Mes con más lesionados (promedio 2018-2025): Marzo
Día con más lesionados: Domingo (17.0 %)
Departamentos que suman el 50 %: 6
Departamento con más fallecidos en sectores críticos: VALLE DEL CAUCA


**Interpretación del perfil** (los valores exactos están en las salidas anteriores):
- **Tendencia:** el pico de la serie es 2016 (77.037 lesionados), seguido de 2017. Después la cifra baja hasta 2020 (27.481, año de pandemia) y se recupera desde 2021; 2023 a 2025 se mantienen entre 47.000 y 51.000 al año.
- **Estacionalidad:** el reparto entre meses es bastante parejo. Marzo tiene el promedio más alto (2018-2025), seguido de octubre, septiembre y agosto; diciembre no está entre los primeros.
- **Día de la semana:** sábado y domingo suman cerca del 33 % de los lesionados (frente a ~28,6 % si fueran como los demás días). El **domingo** es el día con más (17,0 %).
- **Territorio:** la siniestralidad está concentrada. Seis departamentos (Valle del Cauca, Antioquia, Bogotá, Cundinamarca, Santander y Tolima) suman más de la mitad de los lesionados de 2018-2025, y las capitales aportan la mayor parte dentro de cada uno.
- **Sectores críticos:** Valle del Cauca encabeza también los sectores críticos (60 sectores, 684 fallecidos). El sector con el z-score más alto está en Nariño (Tumaco–Pedregal), un departamento que no está entre los primeros en lesionados: los sectores en carretera muestran un riesgo que no se ve en los datos de lesionados.

## 3.3 Reto 3 – 7 ejercicios

**Ejercicio 3.1.** ¿Cuántos lesionados hubo en Antioquia en 2025 y qué municipio aporta más?

In [36]:
ant25 = lesiones[(lesiones["departamento"] == "ANTIOQUIA") & (lesiones["anio"] == 2025)]
print("Total Antioquia 2025:", ant25["cantidad"].sum())
ant25.groupby("municipio", observed=True)["cantidad"].sum().nlargest(5)

Total Antioquia 2025: 4697


municipio
MEDELLIN    2705
BELLO        198
ITAGUI       186
ENVIGADO     144
RIONEGRO     123
Name: cantidad, dtype: int64

**Ejercicio 3.2.** Top 10 de municipios con más lesionados entre 2018 y 2025 (use `query`).

In [37]:
top_muni = (lesiones.query("2018 <= anio <= 2025")
            .groupby(["cod_muni", "municipio", "departamento"], observed=True)["cantidad"].sum()
            .nlargest(10).reset_index(name="lesionados"))
top_muni

,cod_muni,municipio,departamento,lesionados
0,11001,BOGOTA D.C.,BOGOTA D.C.,32275
1,76001,CALI,VALLE DEL CAUCA,23775
2,05001,MEDELLIN,ANTIOQUIA,20354
3,73001,IBAGUE,TOLIMA,14345
4,66001,PEREIRA,RISARALDA,11613
5,50001,VILLAVICENCIO,META,8814
6,68001,BUCARAMANGA,SANTANDER,7575
7,13001,CARTAGENA DE INDIAS,BOLIVAR,6936
8,52001,PASTO,NARIÑO,6852
9,17001,MANIZALES,CALDAS,6759


**Ejercicio 3.3.** Compare enero–septiembre de 2025 contra el mismo periodo de 2026 por departamento y ordene por variación.

In [38]:
ene_sep = lesiones[(lesiones["mes"] <= 9) & lesiones["anio"].isin([2025, 2026])]
comp = ene_sep.pivot_table(index="departamento", columns="anio", values="cantidad", aggfunc="sum", observed=True).fillna(0)
comp["variacion_%"] = (comp[2026] - comp[2025]) / comp[2025] * 100
print("Total nacional ene-sep:", comp[[2025, 2026]].sum().to_dict())
comp.sort_values("variacion_%", ascending=False).round(1).head(10)

Total nacional ene-sep: {2025: 36992, 2026: 41042}


anio,2025,2026,variacion_%
departamento,,,
VICHADA,3,6,100.00
CASANARE,247,475,92.30
SUCRE,429,730,70.20
RISARALDA,1513,2280,50.70
CAUCA,802,1101,37.30
CAQUETA,219,291,32.90
NORTE DE SANTANDER,1038,1352,30.30
CORDOBA,919,1196,30.10
BOLIVAR,1748,2252,28.80


> A nivel nacional, enero–septiembre de 2026 tiene cerca de **11 % más** lesionados que el mismo periodo de 2025. Los aumentos porcentuales más altos aparecen en departamentos con pocos casos (Vichada pasa de 3 a 6), así que conviene leer el porcentaje junto con las cifras absolutas.

**Ejercicio 3.4.** Sectores con 99 % de confianza y 15 o más fallecidos, ordenados de mayor a menor.

In [39]:
criticos = (sectores[(sectores["nivel_confianza"] == "99%") & (sectores["fallecidos"] >= 15)]
            .sort_values(["fallecidos", "gi_zscore"], ascending=False))
print("Sectores:", len(criticos))
criticos[["departamento", "municipio", "tramo", "fallecidos", "gi_zscore", "gi_pvalue"]].head(10)

Sectores: 69


,departamento,municipio,tramo,fallecidos,gi_zscore,gi_pvalue
292,NARIÑO,TUMACO,TUMACO-PEDREGAL,52,14.54,0.00
198,CORDOBA,CERETE,CERETE - SAN PELAYO,43,11.87,0.00
183,ATLANTICO,SOLEDAD,BARRANQUILLA - MALAMBO - PALMAR DE VARELA,35,9.51,0.00
237,VALLE DEL CAUCA,CALI,CALI - JAMUNDI - LA PRIMAVERA,33,8.91,0.00
190,ATLANTICO,MALAMBO,BARRANQUILLA - MALAMBO - PALMAR DE VARELA,33,6.54,0.00
201,ANTIOQUIA,TURBO,CHIGORODO-TURBO,30,8.03,0.00
217,SUCRE,SINCELEJO,PLANETA RICA - SINCELEJO,30,8.03,0.00
215,CORDOBA,MONTERIA,PLANETA RICA - MONTERIA,29,7.73,0.00
95,ANTIOQUIA,COPACABANA,EL HATILLO - GIRARDOTA - COPACABANA - BELLO - MED,29,5.53,0.00
132,VALLE DEL CAUCA,PALMIRA,CALI - PALMIRA,27,7.14,0.00


**Ejercicio 3.5.** Resumen por entidad responsable: número de sectores, fallecidos totales y promedio, y z-score medio.

In [40]:
sectores.groupby("entidad_nombre", observed=True).agg(
    sectores=("fallecidos", "size"), fallecidos=("fallecidos", "sum"),
    fallecidos_prom=("fallecidos", "mean"), zscore_medio=("gi_zscore", "mean"),
).sort_values("fallecidos", ascending=False)

,sectores,fallecidos,fallecidos_prom,zscore_medio
entidad_nombre,,,,
INVÍAS y otras vías nacionales,126,1513,12.01,3.24
Agencia Nacional de Infraestructura,137,1480,10.80,3.04
Vía departamental,48,510,10.62,3.45
Vía urbana,1,12,12.00,2.70


**Ejercicio 3.6.** ¿Qué municipio concentra el mayor porcentaje de lesionados dentro de su departamento (2018-2025)? Use `transform` e `idxmax`.

In [41]:
mun = (recientes.groupby(["departamento", "municipio"], observed=True)["cantidad"].sum().reset_index())
mun["%_en_depto"] = mun["cantidad"] / mun.groupby("departamento", observed=True)["cantidad"].transform("sum") * 100
lider = mun.loc[mun.groupby("departamento", observed=True)["%_en_depto"].idxmax()]
lider.sort_values("%_en_depto", ascending=False).head(10)

,departamento,municipio,cantidad,%_en_depto
0,AMAZONAS,LETICIA,131,100.00
154,BOGOTA D.C.,BOGOTA D.C.,32275,100.00
601,GUAINIA,INIRIDA,133,100.00
605,GUAVIARE,SAN JOSE DEL GUAVIARE,592,97.05
851,SAN ANDRES ISLAS,SAN ANDRES,575,95.20
1050,VAUPES,MITU,40,93.02
346,CAQUETA,FLORENCIA,1513,75.42
1054,VICHADA,PUERTO CARREÑO,24,72.73
449,CHOCO,QUIBDO,687,69.96
750,NARIÑO,PASTO,6852,69.65


> En los departamentos con un solo municipio grande (Amazonas, Guainía, Bogotá) la capital concentra casi el 100 %. Entre los departamentos grandes destacan Pasto en Nariño (~70 %) y Quibdó en Chocó (~70 %).

**Ejercicio 3.7.** Calcule el promedio móvil de 3 años del total nacional y el año con mayor lesionados por cada día de registro.

In [42]:
serie = perfil_anual[["lesionados"]].copy()
serie["media_movil_3"] = serie["lesionados"].rolling(3).mean()
serie["lesionados_por_dia"] = serie["lesionados"] / completos.groupby("anio")["fecha"].nunique()
print("Año con más lesionados por día:", serie["lesionados_por_dia"].idxmax())
serie.tail(8)

Año con más lesionados por día: 2016


,lesionados,media_movil_3,lesionados_por_dia
anio,,,
2018,48999,"64,439.00",134.24
2019,40150,"52,143.33",110.00
2020,27481,"38,876.67",75.08
2021,34114,"33,915.00",93.46
2022,45760,"35,785.00",125.37
2023,51338,"43,737.33",140.65
2024,47344,"48,147.33",129.36
2025,48432,"49,038.00",132.69


### Guardar resultados agregados

In [43]:
perfil_anual.to_csv(PROCESSED / "lesiones_by_year.csv")
(completos.groupby(["anio", "cod_depto", "departamento"], observed=True)["cantidad"].sum()
          .reset_index(name="lesionados")
          .to_csv(PROCESSED / "lesiones_by_department_year.csv", index=False))
sorted(p.name for p in PROCESSED.glob("*.csv"))

['cleaning_log.csv',
 'dim_departments.csv',
 'dim_municipalities.csv',
 'lesiones_by_department_year.csv',
 'lesiones_by_year.csv',
 'lesiones_clean.csv',
 'municipality_analysis.csv',
 'sectores_clean.csv']

---
# Lección 4. Combinación de datasets
## 4.1 Ejemplos de la lección
Para combinar se construyen tablas de dimensión a partir de los datos limpios (como `products`, `aisles` y `departments` en Instacart):
- `dim_departamentos`: `cod_depto` → `departamento`
- `dim_municipios`: `cod_muni` → `municipio`, `cod_depto`

In [44]:
dim_departamentos = (lesiones.dropna(subset=["cod_depto"])[["cod_depto", "departamento"]]
                     .drop_duplicates().astype({"departamento": "string"})
                     .sort_values("cod_depto").reset_index(drop=True))
dim_municipios = (lesiones.dropna(subset=["cod_muni"])[["cod_muni", "municipio", "cod_depto"]]
                  .drop_duplicates().astype({"municipio": "string"})
                  .sort_values("cod_muni").reset_index(drop=True))
print("Departamentos:", len(dim_departamentos), "| claves únicas:", dim_departamentos["cod_depto"].is_unique)
print("Municipios   :", len(dim_municipios), "| claves únicas:", dim_municipios["cod_muni"].is_unique)
dim_municipios.head()

Departamentos: 33 | claves únicas: True
Municipios   : 1083 | claves únicas: True


,cod_muni,municipio,cod_depto
0,05001,MEDELLIN,05
1,05002,ABEJORRAL,05
2,05021,ALEJANDRIA,05
3,05030,AMAGA,05
4,05031,AMALFI,05


### Ejemplo 1 – `merge` inner con validación de cardinalidad

In [45]:
mun_dep = dim_municipios.merge(dim_departamentos, on="cod_depto", how="inner", validate="many_to_one")
print(len(dim_municipios), "->", len(mun_dep))
mun_dep.head(3)

1083 -> 1083


,cod_muni,municipio,cod_depto,departamento
0,05001,MEDELLIN,05,ANTIOQUIA
1,05002,ABEJORRAL,05,ANTIOQUIA
2,05021,ALEJANDRIA,05,ANTIOQUIA


### Ejemplo 2 – `left`, `right` y `outer` con `indicator=True`

In [46]:
les_2025 = (lesiones[lesiones["anio"] == 2025].groupby("cod_muni")["cantidad"].sum()
            .reset_index(name="lesionados_2025"))
sec_mun = sectores.groupby("cod_muni").size().reset_index(name="sectores")
for how in ["inner", "left", "right", "outer"]:
    r = les_2025.merge(sec_mun, on="cod_muni", how=how, indicator=True)
    print(f"{how:6s}: {len(r):5d} filas ->", r["_merge"].value_counts().to_dict())

inner :   133 filas -> {'both': 133, 'left_only': 0, 'right_only': 0}
left  :   863 filas -> {'left_only': 730, 'both': 133, 'right_only': 0}
right :   133 filas -> {'both': 133, 'left_only': 0, 'right_only': 0}
outer :   863 filas -> {'left_only': 730, 'both': 133, 'right_only': 0}


### Ejemplo 3 – Columnas con el mismo nombre: `suffixes`

In [47]:
r = sectores[["cod_muni", "municipio"]].merge(dim_municipios[["cod_muni", "municipio"]],
                                             on="cod_muni", suffixes=("_sector", "_lesiones"))
r.head(3)

,cod_muni,municipio_sector,municipio_lesiones
0,25178,CHIPAQUE,CHIPAQUE
1,25335,GUAYABETAL,GUAYABETAL
2,25335,GUAYABETAL,GUAYABETAL


### Ejemplo 4 – `concat` vertical y horizontal

In [48]:
a24 = lesiones[lesiones["anio"] == 2024].groupby("departamento", observed=True)["cantidad"].sum()
a25 = lesiones[lesiones["anio"] == 2025].groupby("departamento", observed=True)["cantidad"].sum()
vertical = pd.concat([a24.rename("lesionados").reset_index().assign(anio=2024),
                      a25.rename("lesionados").reset_index().assign(anio=2025)], ignore_index=True)
horizontal = pd.concat([a24.rename("2024"), a25.rename("2025")], axis=1)
print("vertical:", vertical.shape, "| horizontal:", horizontal.shape)
horizontal.head()

vertical: (66, 3) | horizontal: (33, 2)


,2024,2025
departamento,,
AMAZONAS,25,43
ANTIOQUIA,4363,4697
ARAUCA,209,191
ATLANTICO,1978,2042
BOGOTA D.C.,4734,3857


### Ejemplo 5 – `join` por índice

In [49]:
idx_les = recientes.groupby("cod_muni")["cantidad"].sum().rename("lesionados_2018_2025").to_frame()
idx_sec = sectores.groupby("cod_muni")["fallecidos"].sum().rename("fallecidos_sectores").to_frame()
idx_les.join(idx_sec, how="inner").sort_values("fallecidos_sectores", ascending=False).head()

,lesionados_2018_2025,fallecidos_sectores
cod_muni,,
52835,349,134
05088,1746,105
76520,5415,97
76111,2528,92
25286,1709,87


## 4.2 Laboratorio guiado 4 – Tabla de análisis por municipio
*(Equivalente a la “tabla de análisis por producto” de Instacart: una fila por municipio con todas sus métricas.)*

Pasos: 1) métricas de lesiones por municipio, 2) métricas de sectores críticos por municipio, 3) combinar con la dimensión de municipios y departamentos, 4) crear indicadores, 5) validar, 6) guardar.

In [50]:
# 1. Métricas de lesiones por municipio
m_les = completos.groupby("cod_muni").agg(
    lesionados_total=("cantidad", "sum"),
    anios_con_registro=("anio", "nunique"),
    primer_anio=("anio", "min"),
    ultimo_anio=("anio", "max"),
).reset_index()
m_les = m_les.merge(les_2025, on="cod_muni", how="left")
m_les = m_les.merge(recientes.groupby("cod_muni")["cantidad"].sum().reset_index(name="lesionados_2018_2025"),
                    on="cod_muni", how="left")

# 2. Métricas de sectores críticos por municipio
m_sec = sectores.groupby("cod_muni").agg(
    sectores_criticos=("fallecidos", "size"),
    fallecidos_sectores=("fallecidos", "sum"),
    zscore_max=("gi_zscore", "max"),
    sectores_99=("nivel_confianza", lambda s: (s == "99%").sum()),
).reset_index()

# 3. Combinar con las dimensiones
tabla_municipios = (dim_municipios
    .merge(dim_departamentos, on="cod_depto", how="left", validate="many_to_one")
    .merge(m_les, on="cod_muni", how="left", validate="one_to_one")
    .merge(m_sec, on="cod_muni", how="left", validate="one_to_one", indicator="origen_sector"))

# 4. Indicadores
cols_cero = ["lesionados_total", "lesionados_2025", "lesionados_2018_2025", "sectores_criticos",
             "fallecidos_sectores", "sectores_99", "anios_con_registro"]
tabla_municipios[cols_cero] = tabla_municipios[cols_cero].fillna(0).astype(int)
tabla_municipios[["primer_anio", "ultimo_anio"]] = tabla_municipios[["primer_anio", "ultimo_anio"]].astype("Int64")
tabla_municipios["tiene_sector_critico"] = tabla_municipios["origen_sector"] == "both"
tabla_municipios["promedio_anual_2018_2025"] = tabla_municipios["lesionados_2018_2025"] / 8
tabla_municipios["ranking_lesionados"] = tabla_municipios["lesionados_2018_2025"].rank(ascending=False, method="min").astype(int)
tabla_municipios["pct_nacional_2018_2025"] = tabla_municipios["lesionados_2018_2025"] / tabla_municipios["lesionados_2018_2025"].sum() * 100
tabla_municipios = tabla_municipios.drop(columns="origen_sector").sort_values("ranking_lesionados")
tabla_municipios.head(10)

,cod_muni,municipio,cod_depto,departamento,lesionados_total,anios_con_registro,primer_anio,ultimo_anio,lesionados_2025,lesionados_2018_2025,sectores_criticos,fallecidos_sectores,zscore_max,sectores_99,tiene_sector_critico,promedio_anual_2018_2025,ranking_lesionados,pct_nacional_2018_2025
147,11001,BOGOTA D.C.,11,BOGOTA D.C.,91034,23,2003,2025,3857,32275,0,0,NaN,0,False,"4,034.38",1,9.39
987,76001,CALI,76,VALLE DEL CAUCA,55536,23,2003,2025,2734,23775,7,80,8.91,3,True,"2,971.88",2,6.92
0,05001,MEDELLIN,05,ANTIOQUIA,45296,23,2003,2025,2705,20354,0,0,NaN,0,False,"2,544.25",3,5.92
940,73001,IBAGUE,73,TOLIMA,28982,23,2003,2025,2187,14345,1,13,3.00,1,True,"1,793.12",4,4.17
813,66001,PEREIRA,66,RISARALDA,30682,23,2003,2025,1450,11613,2,21,3.31,1,True,"1,451.62",5,3.38
671,50001,VILLAVICENCIO,50,META,17539,23,2003,2025,1185,8814,6,58,3.61,3,True,"1,101.75",6,2.57
827,68001,BUCARAMANGA,68,SANTANDER,30019,23,2003,2025,788,7575,1,3,2.63,1,True,946.88,7,2.20
148,13001,CARTAGENA DE INDIAS,13,BOLIVAR,22127,23,2003,2025,1861,6936,1,9,1.81,0,True,867.00,8,2.02
700,52001,PASTO,52,NARIÑO,15941,23,2003,2025,873,6852,3,29,3.48,2,True,856.50,9,1.99
315,17001,MANIZALES,17,CALDAS,19163,23,2003,2025,1206,6759,0,0,NaN,0,False,844.88,10,1.97


In [51]:
# 5. Validaciones
validaciones = pd.DataFrame([
    ("Una fila por municipio", tabla_municipios["cod_muni"].is_unique),
    ("Filas = municipios de la dimensión", len(tabla_municipios) == len(dim_municipios)),
    ("Total lesionados 2003-2025 se conserva", tabla_municipios["lesionados_total"].sum() == completos.dropna(subset=["cod_muni"])["cantidad"].sum()),
    ("Total lesionados 2025 se conserva", tabla_municipios["lesionados_2025"].sum() == les_2025["lesionados_2025"].sum()),
    ("Todos los sectores quedaron asignados", tabla_municipios["sectores_criticos"].sum() == len(sectores)),
    ("Fallecidos de sectores se conservan", tabla_municipios["fallecidos_sectores"].sum() == sectores["fallecidos"].sum()),
    ("Sin departamentos nulos", tabla_municipios["departamento"].notna().all()),
], columns=["validacion", "resultado"])
assert validaciones["resultado"].all(), "Alguna validación falló"
validaciones

,validacion,resultado
0,Una fila por municipio,True
1,Filas = municipios de la dimensión,True
2,Total lesionados 2003-2025 se conserva,True
3,Total lesionados 2025 se conserva,True
4,Todos los sectores quedaron asignados,True
5,Fallecidos de sectores se conservan,True
6,Sin departamentos nulos,True


In [52]:
# 6. Guardar
tabla_municipios.to_csv(PROCESSED / "municipality_analysis.csv", index=False)
dim_municipios.to_csv(PROCESSED / "dim_municipalities.csv", index=False)
dim_departamentos.to_csv(PROCESSED / "dim_departments.csv", index=False)
print(tabla_municipios.shape)
print("Municipios con sector crítico:", tabla_municipios["tiene_sector_critico"].sum())

(1083, 18)
Municipios con sector crítico: 133


**Lectura de la tabla:** cada municipio tiene en una sola fila sus lesionados (total, 2018-2025 y 2025), su ranking, su participación nacional y, si los tiene, sus sectores críticos y fallecidos. Las validaciones confirman que la combinación **no duplicó ni perdió** registros.

## 4.3 Reto 4 – 7 ejercicios

**Ejercicio 4.1.** Una `sectores` con `dim_municipios` (validación `many_to_one`) y confirme con `indicator` que todos los sectores encuentran su municipio.

In [53]:
sec_dim = sectores.merge(dim_municipios.rename(columns={"municipio": "municipio_dane", "cod_depto": "cod_depto_dim"}),
                         on="cod_muni", how="left", validate="many_to_one", indicator=True)
print(sec_dim["_merge"].value_counts())
print("Filas:", len(sectores), "->", len(sec_dim))
print("cod_depto coherente:", (sec_dim["cod_depto"] == sec_dim["cod_depto_dim"]).all())

_merge
both          312
left_only       0
right_only      0
Name: count, dtype: int64
Filas: 312 -> 312
cod_depto coherente: True


**Ejercicio 4.2.** Identifique los municipios cuyo nombre difiere entre las dos fuentes, aunque el código coincida.

In [54]:
dif = sec_dim.loc[sec_dim["municipio"] != sec_dim["municipio_dane"], ["cod_muni", "municipio", "municipio_dane"]].drop_duplicates()
print("Municipios con nombre distinto:", len(dif))
dif

Municipios con nombre distinto: 3


,cod_muni,municipio,municipio_dane
120,76111,BUGA,GUADALAJARA DE BUGA
259,19532,PATIA (EL BORDO),PATIA
288,52835,TUMACO,SAN ANDRES DE TUMACO


> Esto confirma la decisión de la EV05: el cruce debe hacerse por **código DIVIPOLA**. Por nombre se perderían Buga, Tumaco y Patía (Chachagüí ya coincide gracias a la normalización de la lección 2).

**Ejercicio 4.3.** Clasifique los municipios con `outer join` entre los que tuvieron lesionados en 2025 y los que tienen sectores críticos.

In [55]:
cls = les_2025.merge(sec_mun, on="cod_muni", how="outer", indicator=True)
cls["categoria"] = cls["_merge"].map({"both": "Lesionados 2025 y sector crítico",
                                      "left_only": "Solo lesionados 2025",
                                      "right_only": "Solo sector crítico (sin lesionados 2025)"})
cls = cls.merge(dim_municipios[["cod_muni", "municipio"]], on="cod_muni", how="left")
print(cls["categoria"].value_counts())
cls[cls["_merge"] == "right_only"][["cod_muni", "municipio", "sectores"]]

categoria
Solo lesionados 2025                         730
Lesionados 2025 y sector crítico             133
Solo sector crítico (sin lesionados 2025)      0
Name: count, dtype: int64


,cod_muni,municipio,sectores


> Los 133 municipios con sectores críticos registraron lesionados en 2025; ninguno queda como *solo sector crítico*. En cambio, 730 municipios con lesionados no tienen sectores críticos identificados en la red vial nacional.

**Ejercicio 4.4.** Use `concat` para apilar los resúmenes departamentales de 2024 y 2025 y luego compárelos con `pivot`.

In [56]:
pivote = vertical.pivot(index="departamento", columns="anio", values="lesionados").fillna(0)
pivote["cambio"] = pivote[2025] - pivote[2024]
pivote["cambio_%"] = pivote["cambio"] / pivote[2024] * 100
pivote.sort_values("cambio", ascending=False).round(1).head(8)

anio,2024,2025,cambio,cambio_%
departamento,,,,
RISARALDA,1783,2223,440,24.70
ANTIOQUIA,4363,4697,334,7.70
CESAR,827,1054,227,27.40
BOLIVAR,2122,2323,201,9.50
HUILA,1516,1712,196,12.90
CUNDINAMARCA,3943,4115,172,4.40
MAGDALENA,786,930,144,18.30
SUCRE,404,540,136,33.70


**Ejercicio 4.5.** Construya una tabla por departamento que combine lesionados (2018-2025) y sectores críticos, con un indicador de fallecidos en sectores por cada 1.000 lesionados.

In [57]:
dep_les = recientes.groupby("cod_depto")["cantidad"].sum().reset_index(name="lesionados_2018_2025")
dep_sec = sectores.groupby("cod_depto").agg(sectores=("fallecidos", "size"), fallecidos=("fallecidos", "sum")).reset_index()
tabla_dep = (dim_departamentos.merge(dep_les, on="cod_depto", how="left", validate="one_to_one")
                              .merge(dep_sec, on="cod_depto", how="left", validate="one_to_one"))
tabla_dep[["sectores", "fallecidos"]] = tabla_dep[["sectores", "fallecidos"]].fillna(0).astype(int)
tabla_dep["fallecidos_x_1000_lesionados"] = tabla_dep["fallecidos"] / tabla_dep["lesionados_2018_2025"] * 1000
tabla_dep.sort_values("fallecidos_x_1000_lesionados", ascending=False).round(2).head(10)

,cod_depto,departamento,lesionados_2018_2025,sectores,fallecidos,fallecidos_x_1000_lesionados
8,20,CESAR,4799,22,293,61.05
24,81,ARAUCA,1274,5,53,41.60
13,44,LA GUAJIRA,1990,6,82,41.21
16,52,NARIÑO,9840,19,244,24.80
14,47,MAGDALENA,5625,10,128,22.76
7,19,CAUCA,7363,19,145,19.69
9,23,CORDOBA,9384,14,184,19.61
21,70,SUCRE,3047,4,57,18.71
10,25,CUNDINAMARCA,25253,41,422,16.71
23,76,VALLE DEL CAUCA,49903,60,684,13.71


> El indicador no es una tasa de letalidad (las dos fuentes miden cosas distintas). Sirve para señalar departamentos donde la siniestralidad **en carretera** pesa más que la urbana, como Cesar, Arauca y La Guajira, frente a Bogotá, que tiene muchos lesionados y ningún sector crítico.

**Ejercicio 4.6.** Demuestre por qué se valida la cardinalidad: intente un `merge` `one_to_one` con claves duplicadas y capture el error.

In [58]:
try:
    sectores.merge(dim_municipios, on="cod_muni", validate="one_to_one")
except pd.errors.MergeError as e:
    print("MergeError capturado:", e)
print("Claves repetidas en sectores:", sectores["cod_muni"].duplicated().sum())

MergeError capturado: Merge keys are not unique in left dataset; not a one-to-one merge
Duplicates in left:
 cod_muni
   25335
   50001
   15238
   15238
   25295 ...
Claves repetidas en sectores: 179


**Ejercicio 4.7.** Vuelva a leer los archivos procesados y verifique que conservan forma y tipos (prueba de ida y vuelta).

In [59]:
chequeo = []
for nombre, original, dtypes in [
    ("municipality_analysis.csv", tabla_municipios, {"cod_muni": str, "cod_depto": str}),
    ("sectores_clean.csv", sectores, {"cod_muni": str, "cod_depto": str}),
    ("lesiones_clean.csv", lesiones, {"cod_muni": str, "cod_depto": str}),
]:
    leido = pd.read_csv(PROCESSED / nombre, dtype=dtypes)
    chequeo.append({"archivo": nombre, "filas_ok": len(leido) == len(original),
                    "columnas_ok": list(leido.columns) == list(original.columns),
                    "ceros_DIVIPOLA_ok": leido["cod_muni"].dropna().str.len().eq(5).all()})
pd.DataFrame(chequeo)

,archivo,filas_ok,columnas_ok,ceros_DIVIPOLA_ok
0,municipality_analysis.csv,True,True,True
1,sectores_clean.csv,True,True,True
2,lesiones_clean.csv,True,True,True


---
## Conclusión general
- **Limpieza (L2):** los datos quedaron con tipos correctos, códigos DIVIPOLA de 5 dígitos, texto normalizado y nulos explícitos, sin perder lesionados. Cada cambio quedó registrado en `cleaning_log.csv`.
- **Análisis (L3):** la siniestralidad está concentrada en pocos departamentos y municipios; es mayor los fines de semana y bajó con la pandemia, para volver a subir desde 2021.
- **Combinación (L4):** la `municipality_analysis.csv` integra ambas fuentes por código DIVIPOLA. Está validada (sin duplicados ni pérdidas) y es la base para la siguiente etapa del proyecto (visualización y modelado).

Archivos generados en `data/processed/`: `lesiones_clean.csv`, `sectores_clean.csv`, `cleaning_log.csv`, `lesiones_by_year.csv`, `lesiones_by_department_year.csv`, `dim_municipalities.csv`, `dim_departments.csv`, `municipality_analysis.csv`.